In [4]:
import os
import json
import numpy as np
from tensorflow import keras

# =========================
# CONFIG MẶC ĐỊNH
# =========================
DEFAULT_MODEL_PATH = "best_food_model.keras"
DEFAULT_LABELS_PATH = "best_food_model_labels.json"

# =========================
# CORE ML FUNCTIONS
# =========================
def load_food_classifier(model_path, labels_path):
    """Tải model và danh sách nhãn."""
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Không tìm thấy model tại: {model_path}")
    if not os.path.exists(labels_path):
        raise FileNotFoundError(f"Không tìm thấy file labels tại: {labels_path}")

    model = keras.models.load_model(model_path, compile=False)
    with open(labels_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    class_names = list(meta["class_names"])
    image_size = int(meta.get("image_size", 224))
    return model, class_names, image_size

def predict_top_k(model, class_names, image_size, image_path, top_k=5):
    """Dự đoán và trả về top K kết quả."""
    img = keras.utils.load_img(image_path, target_size=(image_size, image_size))
    x = keras.utils.img_to_array(img)
    x = np.expand_dims(x, axis=0) # Thêm batch dimension
    
    probs = model.predict(x, verbose=0)[0]
    idx = np.argsort(probs)[::-1][:top_k]
    
    return [(class_names[i], float(probs[i])) for i in idx]

# =========================
# MAIN
# =========================
def main():
    # CHỈNH SỬA TẠI ĐÂY: Thay đường dẫn ảnh của bạn vào đây
    IMAGE_INPUT = "Screenshot 2026-04-18 082953.png" 
    
    # Các cấu hình khác nếu cần
    MODEL_PATH = DEFAULT_MODEL_PATH
    LABELS_PATH = DEFAULT_LABELS_PATH
    TOP_K = 3

    # Kiểm tra file ảnh đầu vào
    if not os.path.exists(IMAGE_INPUT):
        print(f"❌ Lỗi: Không tìm thấy ảnh tại '{IMAGE_INPUT}'")
        return

    try:
        print("⏳ Đang tải mô hình...")
        model, class_names, image_size = load_food_classifier(MODEL_PATH, LABELS_PATH)
        
        print(f"🔍 Đang phân tích ảnh: {IMAGE_INPUT}")
        results = predict_top_k(model, class_names, image_size, IMAGE_INPUT, top_k=TOP_K)

        # Xuất kết quả
        print("\n" + "="*30)
        print("📊 KẾT QUẢ DỰ ĐOÁN:")
        print("="*30)
        for rank, (label, score) in enumerate(results, 1):
            print(f"{rank}. {label.ljust(15)} | Độ tin cậy: {score*100:.2f}%")
        print("="*30)

    except Exception as e:
        print(f"❌ Có lỗi xảy ra: {e}")

if __name__ == '__main__':
    main()

⏳ Đang tải mô hình...
🔍 Đang phân tích ảnh: Screenshot 2026-04-18 082953.png

📊 KẾT QUẢ DỰ ĐOÁN:
1. Bun dau mam tom | Độ tin cậy: 96.23%
2. Banh khot       | Độ tin cậy: 0.52%
3. Banh cuon       | Độ tin cậy: 0.39%
